# OCC 看涨/看跌成交量来源资格

本卡只验证官方 CSV 语义、免费历史样本、与 Cboe 同交易所合约总量的精确对账，并以一次 Qlib 日历读取核对交易日。不读取价格收益、拟合或交易账户。

In [ ]:
from pathlib import Path
import os, csv, json, hashlib
import pandas as pd
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
root = Path.cwd().resolve()
while not (root / "SPEC.md").is_file():
    if root.parent == root:
        raise RuntimeError("workspace root missing")
    root = root.parent
card = root / "research/occ_soxx_soxl_put_call_source_20261022"
plan = json.loads((card / "qualification_analysis_plan.json").read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
for name, key in [("plan.json","source_plan_sha256"),("retrieval_receipts.json","first_receipt_sha256"),
                  ("reconciliation_result.json","reconciliation_result_sha256"),
                  ("depth_availability_receipts.json","depth_receipt_sha256"),
                  ("second_day_reconciliation_result.json","second_day_reconciliation_sha256")]:
    assert sha(card / name) == plan[key]
assert sha(root / "data/qlib/soxl/qlib_store_manifest.json") == plan["calendar_manifest_sha256"]
assert sha(root / "data/qlib/soxl/calendars/day.txt") == plan["calendar_file_sha256"]
assert not (card / "run_state.json").exists(), "one frozen attempt already ran"
all_receipts = json.loads((card / "retrieval_receipts.json").read_text())["queries"] + json.loads((card / "depth_availability_receipts.json").read_text())["queries"]
records = {}
for item in all_receipts:
    path = root / item["relative_path"]
    assert sha(path) == item["sha256"] == plan["snapshot_sha256"][item["relative_path"]]
    assert item["http_status"] == 200
    if item["reportDate"] == "20230103":
        assert path.read_bytes() == b"Report date cannot be prior to  2 years\r\n"
        records[(item["symbol"],item["reportDate"])] = []
        continue
    with path.open(newline="") as stream:
        reader = csv.reader(stream)
        header = next(reader)
        assert header == ["quantity","underlying","symbol","actype","porc","exchange","actdate"]
        rows = list(reader)
    for row in rows:
        assert len(row) == 8 and row[7] == ""
        assert row[1] == item["symbol"] and row[3] in ("C","F","M") and row[4] in ("C","P")
        assert row[6] == pd.to_datetime(item["reportDate"], format="%Y%m%d").strftime("%m/%d/%Y")
        assert int(row[0]) >= 0
    records[(item["symbol"],item["reportDate"])] = rows
assert len(records) == 8
assert all(len(records[(symbol,"20230103")]) == 0 for symbol in ("SOXX","SOXL"))
assert all(len(records[(symbol,date)]) > 0 for symbol in ("SOXX","SOXL") for date in ("20241101","20261020","20261021"))
first = json.loads((card / "reconciliation_result.json").read_text())
second = json.loads((card / "second_day_reconciliation_result.json").read_text())
assert all(item["ratio"] == 2 for value in first["symbols"].values() for item in value["matched_cboe_exchanges"].values())
assert all(value["all_four_exchanges_exactly_2x"] for value in second["symbols"].values())
(card / "run_state.json").write_text(json.dumps({"status":"started","plan_sha256":sha(card / "qualification_analysis_plan.json")}) + chr(10))
import qlib
from qlib.data import D
from qlib.workflow import R
qlib.init(provider_uri=str(root / plan["calendar_provider"]), region="us", kernels=1)
cal = pd.DatetimeIndex(D.calendar(start_time="2023-01-03",end_time="2026-10-21",freq="day")).normalize()
assert all(pd.Timestamp(date) in cal for date in ["2023-01-03","2024-11-01","2026-10-20","2026-10-21"])
parsed = {symbol:{date:{"rows":len(records[(symbol,date)]),"put_quantity":sum(int(row[0]) for row in records[(symbol,date)] if row[4]=="P"),"call_quantity":sum(int(row[0]) for row in records[(symbol,date)] if row[4]=="C"),"roots":sorted({row[2] for row in records[(symbol,date)]})} for date in ("20230103","20241101","20261020","20261021")} for symbol in ("SOXX","SOXL")}
result={"status":"source_semantics_qualified_limited_history_PIT_gap", "sampled_dates":parsed,
        "matched_Occ_to_Cboe_ratio":2.0,"matching_symbol_date_exchange_comparisons":16,
        "call_put_classification_present":True,"account_types_are_clearing_sides_not_initiation":True,
        "free_2023_sample_unavailable":True,"server_retro_limit":"Report date cannot be prior to  2 years","2024_11_sample_available":True,"latest_2026_10_21_sample_available":True,
        "historical_first_public_time_qualified":False,"continuous_two_full_year_history_qualified":False,
        "model_fits":0,"accounts":0,"market_outcome_reads":0,"D_calendar_calls":1,
        "boundary":"OCC call/put volume is by contract type, exchange and clearing account side; it is not net directional demand or dealer inventory. A put/call ratio cancels the exact two-sided factor but requires vintage and coverage qualification before prediction."}
tracking=card / "qlib_recorder"
with R.start(experiment_name="occ_soxx_soxl_put_call_source_20261022",uri=str(tracking)):
    rec=R.get_recorder()
    rec.save_objects(artifact_path="sources", qualification_plan=plan, retrieval_receipts=all_receipts)
    rec.save_objects(artifact_path="analysis", qualification=result, exact_parsed_rows=records, first_day_reconciliation=first, second_day_reconciliation=second)
    rec.log_metrics(matched_venue_pairs=16, available_symbol_dates=6, model_fits=0, accounts=0)
    result.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking),recorder_local_dir=str(tracking / str(rec.experiment_id) / rec.id))
artifacts=Path(result["recorder_local_dir"])/"artifacts"
manifest={"recorder_id":result["recorder_id"],"entries":[{"path":str(p.relative_to(card)),"sha256":sha(p),"bytes":p.stat().st_size} for p in sorted(artifacts.rglob("*")) if p.is_file()]}
(card / "verification_manifest.json").write_text(json.dumps(manifest,indent=2)+chr(10))
(card / "source_qualification_result.json").write_text(json.dumps(result,indent=2)+chr(10))
summary={"experiment":"occ_soxx_soxl_put_call_source_20261022","status":result["status"],"decision":"source_viable_for_bounded_history_and_forward_timing_qualification_not_information_screen_yet","result":"source_qualification_result.json","recorder_id":result["recorder_id"],"experiment_id":result["experiment_id"],"recorder_uri":result["recorder_uri"],"recorder_local_dir":result["recorder_local_dir"],"new_model_fits":0,"new_accounts":0,"qualified_forward_days":0,"objective_achieved":False,"artifact_policy":json.loads((card / "plan.json").read_text())["artifact_policy"]}
(card / "summary.json").write_text(json.dumps(summary,indent=2)+chr(10))
(card / "conclusion.md").write_text("# 中文结论\n\nOCC 官方免费批量接口返回 SOXX/SOXL 按看涨/看跌、清算账户类型、交易所和合约根拆分的成交量。2026-10-20 与 10-21 两天、两标的、四个共同交易所共 16 个对照中，OCC 各账户数量和始终恰好是 Cboe 合约总量两倍；此为买卖双方清算记录，不能把账户类型解释为交易发起方向。看涨/看跌比值的共同倍数抵消。\n\n2024-11-01 样本有数据，2023-01-03 请求被服务器明确以早于两年限制拒绝。首次公开时间、连续两完整年度历史和修订稳定性尚未合格。没有读取收益、拟合或账户，不能声称有效因子或稳定跑赢 SOXX。Recorder `"+result["recorder_id"]+"`。下一步在有限可得历史内核查连续覆盖和发布时序，再冻结独立的信息检验。\n")
(card / "run_state.json").write_text(json.dumps({"status":"completed","recorder_id":result["recorder_id"],"plan_sha256":sha(card / "qualification_analysis_plan.json")})+chr(10))
print(json.dumps({"status":result["status"],"recorder_id":result["recorder_id"],"sampled_dates":parsed},indent=2))

## 中文结论

OCC 提供按看涨/看跌分类的量，清算账户类型数量和与 Cboe 合约数有精确两倍关系。服务器明确限制两年历史；首次公开时间仍不足以进行正式因子检验；详细指标与 Recorder 身份见同卡 `conclusion.md`、`summary.json`。